# Goal placement: explicit and verbal subgoals

This notebook only reads saved live results. It makes no API calls. Six validation states were deliberately selected near known goal-placement opportunities in `rl-data`; these are component diagnostics, not six full-board solves. Training demonstrations are from separate layouts.

The explicit comparison changes instructions and adds six unlabelled task-progress examples while preserving the 26 calibrated examples and CLLP. The verbal variant has its own controller, judge, and small calibration.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

REPO_PATH: str | None = None
anchors = [Path(REPO_PATH).expanduser()] if REPO_PATH else [
    candidate for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "CaRL")
]
ROOT = next((path for path in anchors if (path / "carl/icl").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Set REPO_PATH to the CaRL checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from carl.icl.boards import from_rows  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.board_graphics import board_comparison  # noqa: E402

BASE = ROOT / "analysis/luna_icl/campaign_100/goal_progress_probe"
REPORT = BASE / "report"
ACTIONS = ("UP", "DOWN", "LEFT", "RIGHT")
pio.renderers.default = "plotly_mimetype"
print("Saved live data:", BASE)


## Verified training examples: putting a box on a goal

Change `DEMO_INDEX` to inspect another training snippet. The actions are verified **training** witnesses. They are not supplied as a solution to validation queries. The six added examples carry no invented `u` or `d` label.

In [ ]:
demonstrations = json.loads((BASE / "data_training_demonstrations.json").read_text())["demonstrations"]
display(pd.DataFrame([{key: row[key] for key in ("id", "k", "goals_before", "goals_after", "walking_actions", "pushing_actions")} for row in demonstrations]))
DEMO_INDEX = 1
demo = demonstrations[DEMO_INDEX]
print("Training actions:", [ACTIONS[index] for index in demo["actions"]])
board_comparison(from_rows(demo["state"]), from_rows(demo["subgoal"]), ("Training source", "Box placed on goal")).show()


## Actual results and denominators

An exact endpoint hit and a box placed on a goal are separate outcomes. For the verbal variant, a positive judge claim is also separate: a preparation goal may be correctly achieved without putting a box on a goal. Missing runs are never counted as failures.

In [ ]:
explicit = pd.read_csv(REPORT / "explicit.csv")
display(explicit)
verbal_path = REPORT / "verbal.csv"
if verbal_path.exists():
    display(pd.read_csv(verbal_path))
retry_path = REPORT / "verbal_grounded_retry.csv"
if retry_path.exists():
    print("Post-hoc retries of the two observed failures; not an independent retest:")
    display(pd.read_csv(retry_path))
pio.read_json(REPORT / "goal_progress.json").show()


## Inspect the worded goal and every judge decision

The policy sees the source, current board, and description. The judge gets a separate request with those same observable inputs. Neither receives a future validation board or reference solution. The simulator executes moves and independently checks full task completion.

In [ ]:
VERBAL_VARIANT = "grounded_retry"  # "initial" for the original four queries
if VERBAL_VARIANT not in {"initial", "grounded_retry"}:
    raise ValueError("Choose initial or grounded_retry.")
run_directory = "verbal_grounded_retry" if VERBAL_VARIANT == "grounded_retry" else "verbal"
runs = []
for checkpoint_path in sorted((BASE / run_directory).glob("*/checkpoint.json")):
    frozen = json.loads(checkpoint_path.read_text())
    if frozen["provider"] == "openai":
        runs.extend(sorted(checkpoint_path.parent.glob("query_*.json")))
runs = [path for path in runs if ".started." not in path.name]
QUERY_INDEX = 0
STEP_INDEX = 0
if runs:
    saved = json.loads(runs[QUERY_INDEX].read_text())
    print("Record:", runs[QUERY_INDEX])
    if saved["attempts"]:
        attempt = saved["attempts"][0]
        print("Textual subgoal:", attempt["goal"])
        display(pd.DataFrame([{
            "step": index + 1, "action": ACTIONS[row["action"]],
            "judge_reached": row["judgment"]["reached"] if row["judgment"] else None,
            "judge_reason": row["judgment"]["reason"] if row["judgment"] else None,
            "task_solved": row["solved"],
        } for index, row in enumerate(attempt["steps"])]))
        if attempt["steps"]:
            step = attempt["steps"][STEP_INDEX]
            board_comparison(from_rows(step["before"]), from_rows(step["after"]), ("Before action", "Actual simulator endpoint")).show()
    else:
        print("Generator returned no textual goal.")
else:
    print("No completed live verbal query; inspect the saved calibration status.")


## Prompts, calibration and costs

The exact full-board prompts are frozen in `explicit/openai/manifest.json`. Verbal checkpoints contain all prompts and controller demonstrations; `raw_calls.jsonl` preserves full requests and outputs, including judge reasons. Verbal calibration uses a single rollout per target and is exploratory.

In [ ]:
calibration_path = REPORT / "verbal_calibration.csv"
if calibration_path.exists():
    display(pd.read_csv(calibration_path))
display(pd.read_csv(REPORT / "costs.csv"))
summary = json.loads((REPORT / "summary.json").read_text())
print("Separate full-board diagnostic:", summary["full_board_32"])
display(Markdown("[Report and exported figures](../analysis/luna_icl/campaign_100/goal_progress_probe/report/README.md)"))
